In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Set_dihedral_angles)=
# Set dihedral angles

*Setting dihedral angles to target values by rotating covalent blocks.*

The function {func}`molsysmt.structure.set_dihedral_angles` alters the 3D conformation of a molecular system by setting specific dihedral angles to target angular values, rotating the covalently connected atomic block on the movable side of the central bond.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.structure.set_dihedral_angles`.
:::


## Basic usage

Let's show how to set a specific backbone $\phi$ dihedral angle to a new target value in Met-enkephalin:


In [2]:
import molsysmt as msm
import pyunitwizard as puw


In [3]:
molsys = msm.convert(msm.systems['Met-enkephalin']['met_enkephalin.h5msm'])


:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We identify the atom quartets defining backbone $\phi$ angles using {func}`molsysmt.topology.get_dihedral_quartets` and check the initial value of the second $\phi$ angle:


In [4]:
phi_quartets = msm.topology.get_dihedral_quartets(molsys, phi=True)
initial_angle = msm.structure.get_dihedral_angles(molsys, dihedral_quartets=phi_quartets[1])
print('Selected quartet:', phi_quartets[1])
print('Initial phi angle:', puw.convert(initial_angle, to_unit='degrees'))


Selected quartet: [26, 28, 30, 33]
Initial phi angle: [[-179.9997685580422]] degree


We inspect the initial 3D conformation of the peptide:


In [5]:
molsysviewer_htmlfile = '_static/views/tools_structure_set_dihedral_angles_1.html'


In [6]:
msm.view(molsys)


'<iframe src="../../../../_static/views/tools_structure_set_dihedral_angles_1.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

We set the dihedral angle to $+60.0^\circ$ using {func}`molsysmt.structure.set_dihedral_angles`:


In [7]:
molsys_mod = msm.structure.set_dihedral_angles(molsys, dihedral_quartets=phi_quartets[1], angles='60.0 degrees')


We verify that the angle has been updated exactly to the requested value:


In [8]:
updated_angle = msm.structure.get_dihedral_angles(molsys_mod, dihedral_quartets=phi_quartets[1])
print('Updated phi angle:', puw.convert(updated_angle, to_unit='degrees'))


Updated phi angle: [[59.99999999980902]] degree


Visualizing the modified conformation in 3D illustrates the conformational rotation of the covalent block:


In [9]:
molsysviewer_htmlfile = '_static/views/tools_structure_set_dihedral_angles_2.html'


In [10]:
msm.view(molsys_mod)


'<iframe src="../../../../_static/views/tools_structure_set_dihedral_angles_2.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Setting multiple dihedral angles simultaneously

Multiple dihedral angles can be set concurrently by passing an array of quartets and target angles:


In [11]:
target_angles = puw.quantity([-60.0, -60.0, -60.0, -60.0], 'degrees')
molsys_multi = msm.structure.set_dihedral_angles(molsys, dihedral_quartets=phi_quartets, angles=target_angles)
all_phi = msm.structure.get_dihedral_angles(molsys_multi, dihedral_quartets=phi_quartets)
print('All phi angles after multi-set:', puw.convert(all_phi, to_unit='degrees'))


All phi angles after multi-set: [[-60.00000000345287 -60.00000000019093 -59.999999995397786  -60.00000000180088]] degree


## Explicit blocks and finite edits

`blocks` is a pool of nonempty atom-index lists. For each quartet, all blocks
containing its fourth atom contribute to the moving population. Every fourth
atom must belong to a block. This pool is shared across quartets; it is not an
array assigning one block to each quartet. `blocks=None` computes the covalent
blocks after cutting the central bond. A custom restricted population changes
which atoms move and can change distances across its boundary.

Both editors reject NaN, infinity, complex values, empty requests and incompatible
angular shapes before publishing coordinates. Size-one angular axes broadcast;
a vector specifies values per quartet and a column matrix specifies values per
structure. Rejection leaves coordinates, box, topology and interaction coverage
unchanged. A successful edit makes interactions on edited structures unevaluated,
including previously evaluated empty structures; no detector runs automatically.
The explicit-block route also accepts coordinate-only forms.


In [12]:
import numpy as np

quartet = np.asarray(phi_quartets[1], dtype=int)
moving_block = sorted(next(
    block for block in msm.topology.get_covalent_blocks(
        molsys, remove_bonds=quartet[1:3]
    ) if quartet[3] in block
))
before_edit = puw.get_value(msm.get(molsys, coordinates=True), to_unit='nm').copy()
molsys_explicit = msm.structure.set_dihedral_angles(
    molsys, dihedral_quartets=quartet, angles='15 degrees',
    blocks=[moving_block], structure_indices=[0], pbc=False
)
try:
    msm.structure.set_dihedral_angles(
        molsys, dihedral_quartets=quartet, angles=puw.quantity(np.nan, 'radians'),
        blocks=[moving_block], structure_indices=[0], pbc=False, in_place=True
    )
except msm.ArgumentError:
    print('Nonfinite edit rejected before coordinate publication.')
else:
    raise AssertionError('The edit should reject a nonfinite request.')
np.testing.assert_array_equal(
    puw.get_value(msm.get(molsys, coordinates=True), to_unit='nm'), before_edit
)

Nonfinite edit rejected before coordinate publication.


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Get dihedral quartets <Tutorial_Get_dihedral_quartets>`: Identify atom quartets for dihedral angles with {func}`molsysmt.topology.get_dihedral_quartets`.
- {ref}`Get dihedral angles <Tutorial_Get_dihedral_angles>`: Measure dihedral angle values with {func}`molsysmt.structure.get_dihedral_angles`.
- {ref}`View <Tutorial_View>`: Render molecular systems interactively with {func}`molsysmt.basic.view`.
- {ref}`Shift dihedral angles <Tutorial_Shift_dihedral_angles>`: Increment or decrement dihedral angles with {func}`molsysmt.structure.shift_dihedral_angles`.
:::
